# PORTUWiC-BR Data Construction Pipelines

This notebook contains three independent candidate-generation pipelines followed by final consolidation. Every output is a candidate resource and requires linguistic validation before release.


## Shared Setup

The cells below define shared paths and reusable lexical helpers.


In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'scripts':
    PROJECT_ROOT = PROJECT_ROOT.parent

INPUT_DIR = PROJECT_ROOT / 'scripts' / 'input'
OUTPUT_DIR = PROJECT_ROOT / 'scripts' / 'output'
DATA_DIR = PROJECT_ROOT / 'scripts' / 'data'
for directory in (INPUT_DIR, OUTPUT_DIR, DATA_DIR):
    directory.mkdir(parents=True, exist_ok=True)


## Pipeline 1. Ordinary PT-BR WiC Candidates

This pipeline combines OpenWordNet-PT with senses in the local Portuguese Kaikki/Wiktionary dump that are marked as Brazilian Portuguese. It retains only authentic example contexts and generates balanced same-sense and different-sense candidates.


In [2]:
#!/usr/bin/env python3
"""Generate prototype WiC-style candidates from OpenWordNet-PT.

This script is intentionally conservative: OpenWordNet-PT is useful for finding
Portuguese lemmas, synsets, glosses, and some examples, but it is not a corpus of
natural sentence pairs. The generated rows are candidates for inspection, not a
gold WiC dataset.
"""

from __future__ import annotations

import argparse
import csv
import gzip
import hashlib
import json
import random
import random
import re
import textwrap
import urllib.request
from collections import defaultdict
from dataclasses import dataclass, field
from pathlib import Path


OPENWN_FILES = {
    "wn-aa.jsonl": "https://raw.githubusercontent.com/own-pt/openWordnet-PT/master/dump/wn-aa.jsonl",
    "wn-ab.jsonl": "https://raw.githubusercontent.com/own-pt/openWordnet-PT/master/dump/wn-ab.jsonl",
    "wn-ac.jsonl": "https://raw.githubusercontent.com/own-pt/openWordnet-PT/master/dump/wn-ac.jsonl",
    "own-pt-synsets.ttl": "https://raw.githubusercontent.com/own-pt/openWordnet-PT/master/data/own-pt-synsets.ttl",
}

KAIKKI_FILES = {
    "ptwiktionary.jsonl.gz": "https://kaikki.org/ptwiktionary/raw-wiktextract-data.jsonl.gz",
}


@dataclass
class Synset:
    synset_id: str
    pos: str
    source: str = "OpenWordNet-PT"
    lemmas_pt: list[str] = field(default_factory=list)
    lemmas_en: list[str] = field(default_factory=list)
    gloss_en: list[str] = field(default_factory=list)
    gloss_pt: list[str] = field(default_factory=list)
    examples_pt: list[str] = field(default_factory=list)


def download(url: str, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and path.stat().st_size > 0:
        return
    print(f"Downloading {url}")
    with urllib.request.urlopen(url, timeout=120) as response:
        path.write_bytes(response.read())


def ensure_openwn_files(data_dir: Path) -> dict[str, Path]:
    paths = {}
    for filename, url in OPENWN_FILES.items():
        path = data_dir / filename
        download(url, path)
        paths[filename] = path
    return paths


def parse_quoted_values(block: str, predicate: str) -> list[str]:
    values = []
    pattern = re.compile(rf"{re.escape(predicate)}\s+(.+?)(?:\s*;\n|\s*\.\n)", re.DOTALL)
    match = pattern.search(block)
    if not match:
        return values
    object_text = match.group(1)
    for value in re.findall(r'"((?:[^"\\]|\\.)*)"\@pt', object_text):
        values.append(value.replace('\\"', '"').replace("\\n", " ").strip())
    return values


def parse_synset_ttl(path: Path) -> dict[str, dict[str, list[str]]]:
    text = path.read_text(encoding="utf-8")
    blocks = re.split(r"\n(?=own-pt:synset-)", text)
    parsed: dict[str, dict[str, list[str]]] = {}
    for block in blocks:
        syn_match = re.search(r"own-pt:(synset-[0-9]+-[a-z])\b", block)
        if not syn_match:
            continue
        synset_id = syn_match.group(1).replace("synset-", "")
        parsed[synset_id] = {
            "gloss_pt": parse_quoted_values(block, "owns:gloss"),
            "examples_pt": parse_quoted_values(block, "owns:example"),
        }
    return parsed


def load_synsets(data_dir: Path) -> list[Synset]:
    paths = ensure_openwn_files(data_dir)
    pt_metadata = parse_synset_ttl(paths["own-pt-synsets.ttl"])
    synsets: dict[str, Synset] = {}

    for filename in ("wn-aa.jsonl", "wn-ab.jsonl", "wn-ac.jsonl"):
        with paths[filename].open(encoding="utf-8") as fh:
            for line in fh:
                if not line.strip():
                    continue
                row = json.loads(line)
                src = row.get("_source", {})
                synset_id = src.get("doc_id")
                lemmas_pt = sorted(set(src.get("word_pt", [])))
                if not synset_id or not lemmas_pt:
                    continue
                pos = synset_id.rsplit("-", 1)[-1]
                metadata = pt_metadata.get(synset_id, {})
                synsets[synset_id] = Synset(
                    synset_id=synset_id,
                    pos=pos,
                    source="OpenWordNet-PT",
                    lemmas_pt=lemmas_pt,
                    lemmas_en=sorted(set(src.get("word_en", []))),
                    gloss_en=src.get("gloss_en", []),
                    gloss_pt=metadata.get("gloss_pt", []),
                    examples_pt=metadata.get("examples_pt", []),
                )
    return list(synsets.values())


def ensure_kaikki_files(data_dir: Path) -> dict[str, Path]:
    paths = {}
    for filename, url in KAIKKI_FILES.items():
        path = data_dir / filename
        download(url, path)
        paths[filename] = path
    return paths


def map_kaikki_pos(pos: str) -> str:
    mapping = {
        "noun": "n",
        "verb": "v",
        "adj": "a",
        "adv": "r",
    }
    return mapping.get(pos, pos)


def extract_kaikki_examples(sense: dict, lemma: str) -> list[str]:
    examples = []
    for example in sense.get("examples", []) or []:
        text = example.get("text", "").strip()
        if text and find_target_span(text, lemma):
            examples.append(text)
    return examples


def is_inflectional_sense(glosses: list[str], sense: dict) -> bool:
    text = " ".join(glosses + sense.get("tags", []) + sense.get("raw_tags", [])).lower()
    patterns = (
        "primeira pessoa",
        "segunda pessoa",
        "terceira pessoa",
        "plural de ",
        "feminino de ",
        "masculino de ",
        "forma de ",
        "flexão de ",
        "particípio",
        "gerúndio",
    )
    return any(pattern in text for pattern in patterns)


def load_kaikki_senses(data_dir: Path, require_brazilian_hint: bool = False) -> list[Synset]:
    paths = ensure_kaikki_files(data_dir)
    senses: list[Synset] = []
    with gzip.open(paths["ptwiktionary.jsonl.gz"], "rt", encoding="utf-8") as fh:
        for line in fh:
            if not line.strip():
                continue
            row = json.loads(line)
            if row.get("lang_code") != "pt":
                continue
            word = clean_lemma(row.get("word", ""))
            if not word:
                continue
            if require_brazilian_hint and not entry_has_brazilian_hint(row):
                continue
            pos = map_kaikki_pos(row.get("pos", ""))
            for idx, sense in enumerate(row.get("senses", []) or [], start=1):
                examples = extract_kaikki_examples(sense, word)
                glosses = [g.strip() for g in sense.get("glosses", []) or [] if g.strip()]
                if is_inflectional_sense(glosses, sense):
                    continue
                if not glosses and not examples:
                    continue
                senses.append(
                    Synset(
                        synset_id=f"kaikki:{word}:{pos}:{idx}",
                        pos=pos,
                        source="Kaikki/Wiktionary",
                        lemmas_pt=[word],
                        gloss_pt=glosses,
                        examples_pt=examples,
                    )
                )
    return senses


def entry_has_brazilian_hint(row: dict) -> bool:
    haystack = json.dumps(
        {
            "tags": row.get("tags", []),
            "raw_tags": row.get("raw_tags", []),
            "categories": row.get("categories", []),
            "sounds": row.get("sounds", []),
            "senses": row.get("senses", []),
        },
        ensure_ascii=False,
    ).lower()
    return any(token in haystack for token in ("brazil", "brasil", "brasileir", "carioca", "paulista", "caipira"))


def clean_lemma(lemma: str) -> str:
    return lemma.replace("_", " ").strip()


def keep_lemma(lemma: str, allow_multiword: bool) -> bool:
    cleaned = clean_lemma(lemma)
    if not cleaned or any(char.isdigit() for char in cleaned):
        return False
    if not allow_multiword and re.search(r"\s", cleaned):
        return False
    return True


def find_target_span(sentence: str, lemma: str) -> tuple[int, int] | None:
    cleaned = clean_lemma(lemma)
    pattern = re.compile(rf"(?<!\w){re.escape(cleaned)}(?!\w)", flags=re.IGNORECASE)
    match = pattern.search(sentence)
    if not match:
        return None
    return match.start(), match.end()


def count_words(sentence: str) -> int:
    return len(re.findall(r"\w+", sentence, flags=re.UNICODE))


def mark_target(sentence: str, start: int, end: int) -> str:
    return f"{sentence[:start]}<target>{sentence[start:end]}</target>{sentence[end:]}"


def make_context(
    sentence: str,
    lemma: str,
    context_type: str,
    min_context_words: int = 4,
) -> dict[str, str | int] | None:
    if context_type == "example" and count_words(sentence) < min_context_words:
        return None
    span = find_target_span(sentence, lemma)
    if span is None:
        return None
    start, end = span
    return {
        "sentence": sentence,
        "target_start": start,
        "target_end": end,
        "sentence_marked": mark_target(sentence, start, end),
        "context_type": context_type,
    }


def template_context(lemma: str, description: str, context_type: str) -> dict[str, str | int]:
    cleaned = clean_lemma(lemma)
    sentence = f"{cleaned} é usado neste contexto no sentido de {description}."
    context = make_context(sentence, cleaned, context_type)
    if context is None:
        raise ValueError(f"Could not locate target {cleaned!r} in generated template.")
    return context


def context_for(
    synset: Synset,
    lemma: str,
    allow_templates: bool,
    min_context_words: int,
) -> dict[str, str | int] | None:
    for example in synset.examples_pt:
        context = make_context(example, lemma, "example", min_context_words=min_context_words)
        if context is not None:
            return context
    if not allow_templates:
        return None
    if synset.gloss_pt:
        return template_context(lemma, synset.gloss_pt[0], "gloss_pt_template")
    if synset.gloss_en:
        return template_context(lemma, f"aproximadamente {synset.gloss_en[0]}", "gloss_en_template")
    return template_context(lemma, f"o synset {synset.synset_id}", "fallback_template")


def exact_example_contexts(
    synset: Synset,
    lemma: str,
    min_context_words: int,
) -> list[dict[str, str | int]]:
    contexts = []
    for example in synset.examples_pt:
        context = make_context(example, lemma, "example", min_context_words=min_context_words)
        if context is not None:
            contexts.append(context)
    return contexts


def row_for_pair(
    lemma: str,
    pos: str,
    context1: dict[str, str | int],
    context2: dict[str, str | int],
    label: str,
    first: Synset,
    second: Synset,
) -> dict[str, str]:
    return {
        "target_word": lemma,
        "pos": pos,
        "sentence1": str(context1["sentence"]),
        "sentence2": str(context2["sentence"]),
        "target1_start": str(context1["target_start"]),
        "target1_end": str(context1["target_end"]),
        "target2_start": str(context2["target_start"]),
        "target2_end": str(context2["target_end"]),
        "sentence1_marked": str(context1["sentence_marked"]),
        "sentence2_marked": str(context2["sentence_marked"]),
        "label": label,
        "sense1_id": first.synset_id,
        "sense2_id": second.synset_id,
        "sense1_gloss_pt": " | ".join(first.gloss_pt),
        "sense2_gloss_pt": " | ".join(second.gloss_pt),
        "sense1_gloss_en": " | ".join(first.gloss_en),
        "sense2_gloss_en": " | ".join(second.gloss_en),
        "context1_type": str(context1["context_type"]),
        "context2_type": str(context2["context_type"]),
        "source": first.source if first.source == second.source else f"{first.source}+{second.source}",
        "notes": "Automatically generated candidate; requires linguistic validation.",
    }


def build_candidates(
    synsets: list[Synset],
    max_pairs: int,
    seed: int,
    allow_multiword: bool = False,
    allow_templates: bool = False,
    allow_missing_pt_gloss: bool = False,
    balance: bool = True,
    min_context_words: int = 4,
) -> list[dict[str, str]]:
    rng = random.Random(seed)
    by_lemma_pos: dict[tuple[str, str], list[Synset]] = defaultdict(list)
    for synset in synsets:
        for lemma in synset.lemmas_pt:
            if not keep_lemma(lemma, allow_multiword):
                continue
            by_lemma_pos[(clean_lemma(lemma).lower(), synset.pos)].append(synset)

    positive_rows = []
    positive_items = [
        (clean_lemma(lemma).lower(), synset.pos, synset)
        for synset in synsets
        for lemma in synset.lemmas_pt
        if keep_lemma(lemma, allow_multiword)
        and len(exact_example_contexts(synset, lemma, min_context_words=min_context_words)) >= 2
    ]
    rng.shuffle(positive_items)
    positive_items.sort(key=lambda item: item[0])

    for lemma, pos, synset in positive_items:
        contexts = exact_example_contexts(synset, lemma, min_context_words=min_context_words)
        if contexts[0]["sentence"] == contexts[1]["sentence"]:
            continue
        positive_rows.append(row_for_pair(lemma, pos, contexts[0], contexts[1], "T", synset, synset))

    ambiguous_items = [
        (lemma, pos, senses)
        for (lemma, pos), senses in by_lemma_pos.items()
        if len({s.synset_id for s in senses}) >= 2
        and (
            allow_templates
            or sum(
                1
                for s in {sense.synset_id: sense for sense in senses}.values()
                if exact_example_contexts(s, lemma, min_context_words=min_context_words)
            )
            >= 2
        )
        and (
            allow_missing_pt_gloss
            or sum(1 for s in {sense.synset_id: sense for sense in senses}.values() if s.gloss_pt) >= 2
        )
    ]
    rng.shuffle(ambiguous_items)
    ambiguous_items.sort(
        key=lambda item: (
            not all(s.gloss_pt or s.examples_pt for s in item[2]),
            not any(s.examples_pt for s in item[2]),
            item[0],
        )
    )

    negative_rows = []
    for lemma, pos, senses in ambiguous_items:
        senses = sorted({s.synset_id: s for s in senses}.values(), key=lambda s: s.synset_id)
        if not allow_templates:
            senses = [
                sense
                for sense in senses
                if exact_example_contexts(sense, lemma, min_context_words=min_context_words)
            ]
        if not allow_missing_pt_gloss:
            senses = [sense for sense in senses if sense.gloss_pt]
        for first, second in zip(senses, senses[1:]):
            c1 = context_for(
                first,
                lemma,
                allow_templates=allow_templates,
                min_context_words=min_context_words,
            )
            c2 = context_for(
                second,
                lemma,
                allow_templates=allow_templates,
                min_context_words=min_context_words,
            )
            if c1 is None or c2 is None:
                continue
            if c1["sentence"] == c2["sentence"]:
                continue
            negative_rows.append(row_for_pair(lemma, pos, c1, c2, "F", first, second))

    if balance:
        per_label = min(max_pairs // 2, len(positive_rows), len(negative_rows))
        rows = positive_rows[:per_label] + negative_rows[:per_label]
    else:
        rows = positive_rows + negative_rows
        rows = rows[:max_pairs]
    rng.shuffle(rows)
    return rows


def write_csv(rows: list[dict[str, str]], output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        output_path.write_text("", encoding="utf-8")
        return
    with output_path.open("w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


def main() -> None:
    parser = argparse.ArgumentParser(
        description="Generate a small WiC-style candidate CSV from OpenWordNet-PT."
    )
    parser.add_argument("--data-dir", type=Path, default=Path("scripts/data/openwordnet_pt"))
    parser.add_argument("--kaikki-dir", type=Path, default=Path("scripts/data/kaikki"))
    parser.add_argument("--output", type=Path, default=Path("scripts/output/wic_candidates.csv"))
    parser.add_argument("--max-pairs", type=int, default=50)
    parser.add_argument("--seed", type=int, default=13)
    parser.add_argument("--min-context-words", type=int, default=4)
    parser.add_argument(
        "--sources",
        nargs="+",
        choices=("openwn", "kaikki"),
        default=("openwn", "kaikki"),
        help="Lexical sources to use.",
    )
    parser.add_argument("--allow-multiword", action="store_true")
    parser.add_argument(
        "--allow-templates",
        action="store_true",
        help="Allow gloss-derived template contexts when no natural example contains the target.",
    )
    parser.add_argument(
        "--allow-missing-pt-gloss",
        action="store_true",
        help="Allow negative pairs where one or both synsets lack Portuguese glosses.",
    )
    parser.add_argument("--no-balance", action="store_true", help="Do not balance T and F labels.")
    parser.add_argument(
        "--kaikki-brazilian-hint-only",
        action="store_true",
        help="Use only Kaikki/Wiktionary entries with Brazilian/regional hints.",
    )
    args = parser.parse_args()

    synsets = []
    if "openwn" in args.sources:
        synsets.extend(load_synsets(args.data_dir))
    if "kaikki" in args.sources:
        synsets.extend(
            load_kaikki_senses(
                args.kaikki_dir,
                require_brazilian_hint=args.kaikki_brazilian_hint_only,
            )
        )
    rows = build_candidates(
        synsets,
        max_pairs=args.max_pairs,
        seed=args.seed,
        allow_multiword=args.allow_multiword,
        allow_templates=args.allow_templates,
        allow_missing_pt_gloss=args.allow_missing_pt_gloss,
        balance=not args.no_balance,
        min_context_words=args.min_context_words,
    )
    write_csv(rows, args.output)

    print(f"Loaded {len(synsets)} Portuguese senses/synsets with at least one PT lemma.")
    print(f"Wrote {len(rows)} candidate pairs to {args.output}.")
    print()
    for row in rows[:5]:
        print(textwrap.dedent(
            f"""
            target={row['target_word']} pos={row['pos']} label={row['label']}
            s1={row['sentence1']}
            s2={row['sentence2']}
            sense1={row['sense1_id']} | {row['sense1_gloss_pt'] or row['sense1_gloss_en']}
            sense2={row['sense2_id']} | {row['sense2_gloss_pt'] or row['sense2_gloss_en']}
            """
        ).strip())
        print("-" * 80)


In [3]:
import random

import pandas as pd
# Pipeline 1 output: ordinary Brazilian Portuguese WiC candidates.
openwn_dir = DATA_DIR / 'openwordnet_pt'
kaikki_pt_dir = DATA_DIR / 'kaikki'

normal_senses = []
normal_senses.extend(load_synsets(openwn_dir))
# Restrict Wiktionary-derived senses to entries marked as Brazilian Portuguese.
normal_senses.extend(load_kaikki_senses(kaikki_pt_dir, require_brazilian_hint=True))

normal_rows = build_candidates(
    normal_senses,
    max_pairs=1000,
    seed=13,
    allow_multiword=False,
    allow_templates=False,
    allow_missing_pt_gloss=False,
    balance=True,
    min_context_words=4,
)
normal_path = OUTPUT_DIR / 'pipeline1_normal_ptbr_wic.csv'
write_csv(normal_rows, normal_path)
normal_df = pd.read_csv(normal_path)
print(f'Wrote {len(normal_df)} Pipeline 1 candidates to {normal_path.name}.')
normal_df['label'].value_counts(), normal_df.head(3)


Wrote 558 Pipeline 1 candidates to pipeline1_normal_ptbr_wic.csv.


(label
 T    279
 F    279
 Name: count, dtype: int64,
   target_word pos                                          sentence1  \
 0     criação   n  Ele cresceu rodeado pela criação de cabras e o...   
 1   deleriado   a                         João saiu daqui deleriado.   
 2     dedicar   v            dedicar seus talentos para um boa causa   
 
                                            sentence2  target1_start  \
 0  Josefina sai cá fora e vem vê / Olha os forro ...             25   
 1                              Paulo é um deleriado.             16   
 2                          dedicar sua vida à igreja              0   
 
    target1_end  target2_start  target2_end  \
 0           32             90           97   
 1           25             11           20   
 2            7              0            7   
 
                                     sentence1_marked  \
 0  Ele cresceu rodeado pela <target>criação</targ...   
 1        João saiu daqui <target>deleriado</target>.   


## Pipeline 2. Regional PT-BR WiC Candidates

This pipeline first extracts high-confidence regional lexical seeds from the local Portuguese Kaikki/Wiktionary dump and then uses an LLM to generate natural contexts for linguistic validation.


In [4]:
import csv
import gzip
import json
import re
import urllib.parse
import unicodedata
from collections import defaultdict
from pathlib import Path

from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'scripts':
    PROJECT_ROOT = PROJECT_ROOT.parent

KAIKKI_PATH = PROJECT_ROOT / 'scripts' / 'data' / 'kaikki' / 'ptwiktionary.jsonl.gz'
REGIONAL_ATTESTED_PATH = PROJECT_ROOT / 'scripts' / 'input' / 'regional_terms_kaikki_geographic_attested.csv'
REGIONAL_SEED_PATH = PROJECT_ROOT / 'scripts' / 'input' / 'regional_words_kaikki_geographic_500.csv'
RUN_KAIKKI_REGIONAL_EXTRACTION = True

GEOGRAPHIC_LABELS = {
    'Northeast-Brazil': ('macroregion', 'Nordeste'), 'North-Brazil': ('macroregion', 'Norte'),
    'Southern-Brazil': ('macroregion', 'Sul'), 'Southeast-Brazil': ('macroregion', 'Sudeste'),
    'Central-West-Brazil': ('macroregion', 'Centro-Oeste'), 'Acre': ('state', 'Acre'),
    'Alagoas': ('state', 'Alagoas'), 'Amapá': ('state', 'Amapá'), 'Amazonas': ('state', 'Amazonas'),
    'Bahia': ('state', 'Bahia'), 'Ceará': ('state', 'Ceará'), 'Distrito Federal': ('federal_district', 'Distrito Federal'),
    'Espírito Santo': ('state', 'Espírito Santo'), 'Goiás': ('state', 'Goiás'),
    'Maranhão': ('state', 'Maranhão'), 'Mato Grosso': ('state', 'Mato Grosso'),
    'Mato Grosso do Sul': ('state', 'Mato Grosso do Sul'), 'Minas Gerais': ('state', 'Minas Gerais'),
    'Pará': ('state', 'Pará'), 'Paraíba': ('state', 'Paraíba'), 'Paraná': ('state', 'Paraná'),
    'Pernambuco': ('state', 'Pernambuco'), 'Piauí': ('state', 'Piauí'),
    'Rio de Janeiro': ('state', 'Rio de Janeiro'), 'Rio Grande do Norte': ('state', 'Rio Grande do Norte'),
    'Rio Grande do Sul': ('state', 'Rio Grande do Sul'), 'Rondônia': ('state', 'Rondônia'),
    'Roraima': ('state', 'Roraima'), 'Santa Catarina': ('state', 'Santa Catarina'),
    'São Paulo': ('state', 'São Paulo'), 'Sergipe': ('state', 'Sergipe'), 'Tocantins': ('state', 'Tocantins'),
}

def clean_kaikki_word(value):
    value = re.sub(r'\s+', ' ', str(value)).strip().lower()
    return value if re.fullmatch(r"[a-zà-ÿ][a-zà-ÿ' -]{0,59}", value) else ''

def geographic_matches(evidence):
    matches = []
    for marker, metadata in GEOGRAPHIC_LABELS.items():
        if marker.casefold() in evidence.casefold():
            matches.append(metadata)
    return list(dict.fromkeys(matches))

def extract_kaikki_regional_terms(path):
    rows = []
    with gzip.open(path, 'rt', encoding='utf-8') as handle:
        for line in tqdm(handle, desc='Extracting local Kaikki regional senses', unit='entry'):
            entry = json.loads(line)
            word = clean_kaikki_word(entry.get('word', ''))
            if entry.get('lang_code') != 'pt' or not word:
                continue
            for sense_index, sense in enumerate(entry.get('senses', []) or [], start=1):
                evidence_items = [str(item) for item in ((sense.get('tags') or []) + (sense.get('raw_tags') or []) + (sense.get('categories') or []))]
                evidence = ' | '.join(evidence_items)
                locations = geographic_matches(evidence)
                if not locations or not any(marker in evidence.casefold() for marker in ('brazil', 'brasil', 'brasileir')):
                    continue
                glosses = [gloss.strip() for gloss in sense.get('glosses', []) or [] if gloss.strip()]
                if not glosses:
                    continue
                rows.append({
                    'word': word,
                    'sense_id': f"kaikki:{word}:{entry.get('pos', '')}:{sense_index}",
                    'pos': entry.get('pos', ''),
                    'regional_label': '; '.join(label for _, label in locations),
                    'regional_level': '; '.join(level for level, _ in locations),
                    'region_hint': '; '.join(label for _, label in locations),
                    'notes': glosses[0],
                    'evidence_excerpt': evidence,
                    'source': 'Kaikki Portuguese Wiktionary dump',
                    'source_url': 'https://pt.wiktionary.org/wiki/' + urllib.parse.quote(word.replace(' ', '_')),
                    'source_revision': '',
                    'source_snapshot': KAIKKI_PATH.name,
                    'evidence_status': 'explicit_geographic_attestation_requires_linguist_review',
                    'linguist_status': 'pending',
                })
    return rows

def fold_text(value):
    return ''.join(character for character in unicodedata.normalize('NFKD', value.casefold()) if not unicodedata.combining(character))

def regionality_score(row):
    evidence = row['evidence_excerpt'].casefold()
    levels = row['regional_level'].split('; ')
    labels = row['regional_label'].split('; ')
    score = 0
    score += 8 if 'regionalismo' in evidence or 'regionalism' in evidence else 0
    score += 6 if 'regionalismo (portugues)' in fold_text(evidence) else 0
    score += 4 * sum(level == 'state' for level in levels)
    score += 3 * sum(level == 'macroregion' for level in levels)
    score += 2 if any(marker in evidence for marker in ('giria', 'coloquial', 'popular')) else 0
    folded_word = re.sub(r'[^a-z0-9]+', ' ', fold_text(row['word'])).strip()
    for label in labels:
        folded_label = re.sub(r'[^a-z0-9]+', ' ', fold_text(label)).strip()
        if folded_label and re.search(rf'(?<!\w){re.escape(folded_label)}(?!\w)', folded_word):
            score -= 7
    return score

def select_high_confidence_regional_seeds(rows, n=500):
    best_by_word = {}
    for row in rows:
        row = dict(row)
        row['regionality_score'] = regionality_score(row)
        previous = best_by_word.get(row['word'])
        if previous is None or row['regionality_score'] > previous['regionality_score']:
            best_by_word[row['word']] = row
    ranked = sorted(best_by_word.values(), key=lambda row: (-row['regionality_score'], row['word']))
    return ranked[:n]

def write_regional_rows(rows, output_path):
    if not rows:
        raise RuntimeError('No explicit geographic regional senses were found in the local Kaikki dump.')
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

if RUN_KAIKKI_REGIONAL_EXTRACTION:
    regional_rows = extract_kaikki_regional_terms(KAIKKI_PATH)
    write_regional_rows(regional_rows, REGIONAL_ATTESTED_PATH)
    regional_seed_rows = select_high_confidence_regional_seeds(regional_rows, n=500)
    write_regional_rows(regional_seed_rows, REGIONAL_SEED_PATH)
    print(f'Wrote {len(regional_rows)} explicit geographic senses and {len(regional_seed_rows)} high-confidence regional seeds from the local Kaikki dump.')
else:
    print('Local Kaikki regional extraction is disabled. Set RUN_KAIKKI_REGIONAL_EXTRACTION = True to run it.')


/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Extracting local Kaikki regional senses: 627516entry [00:03, 161601.89entry/s]

Wrote 1022 explicit geographic senses and 500 high-confidence regional seeds from the local Kaikki dump.


In [5]:
# Pipeline 2B: generate natural PT-BR contexts for the high-confidence regional seeds.
# %pip install openai pandas tqdm python-dotenv

import json
import os
import random
import re
import time

import pandas as pd
from tqdm.auto import tqdm

try:
    from openai import OpenAI
except ImportError:
    OpenAI = None

KAIKKI_SEED_PATH = INPUT_DIR / 'regional_words_kaikki_geographic_500.csv'
REGIONAL_MODEL = 'gpt-5-mini'
REGIONAL_TEMPERATURE = 0.7
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', '')
# OPENAI_API_KEY = 'paste-your-key-here'
RUN_REGIONAL_LLM_GENERATION = False
REGIONAL_PAIRS_PER_LABEL = 100

REGIONAL_SCHEMA = {
    'type': 'object',
    'properties': {
        'sentence1': {'type': 'string'},
        'sentence2': {'type': 'string'},
        'sense1': {'type': 'string'},
        'sense2': {'type': 'string'},
    },
    'required': ['sentence1', 'sentence2', 'sense1', 'sense2'],
    'additionalProperties': False,
}

def target_occurrences(sentence, target):
    return len(re.findall(rf'(?<!\w){re.escape(target)}(?!\w)', sentence, flags=re.IGNORECASE))

def mark_target(sentence, target):
    match = re.search(rf'(?<!\w){re.escape(target)}(?!\w)', sentence, flags=re.IGNORECASE)
    return f'{sentence[:match.start()]}<target>{sentence[match.start():match.end()]}</target>{sentence[match.end():]}'

def generate_regional_row(seed, label, client):
    relation = 'the same meaning' if label == 'T' else 'clearly different meanings'
    prompt = f"""
Create one Brazilian Portuguese Word-in-Context candidate.

Target word: {seed['word']}
Required relation: {relation}
Attested geographic label: {seed['regional_label']}
Source sense note: {seed['notes']}
Source evidence: {seed['evidence_excerpt']}

Write two natural Brazilian Portuguese sentences. Use the exact target word exactly once in each sentence.
For label T, use the attested meaning in both sentences. For label F, make the meanings clearly different.
Do not write definitions, explanations, target tags, or quotation marks around the whole sentence.
""".strip()
    response = client.responses.create(
        model=REGIONAL_MODEL,
        temperature=REGIONAL_TEMPERATURE,
        input=[
            {'role': 'developer', 'content': 'Return only valid JSON. Avoid stereotypes and metalinguistic wording.'},
            {'role': 'user', 'content': prompt},
        ],
        text={'format': {'type': 'json_schema', 'name': 'regional_wic', 'strict': True, 'schema': REGIONAL_SCHEMA}},
    )
    row = json.loads(response.output_text)
    if target_occurrences(row['sentence1'], seed['word']) != 1 or target_occurrences(row['sentence2'], seed['word']) != 1:
        raise ValueError('The regional target word must occur exactly once in each sentence.')
    return row

if RUN_REGIONAL_LLM_GENERATION:
    if OpenAI is None or not OPENAI_API_KEY:
        raise RuntimeError('Set OPENAI_API_KEY and install the OpenAI SDK before regional generation.')
    seeds = pd.read_csv(KAIKKI_SEED_PATH).to_dict(orient='records')
    if len(seeds) < REGIONAL_PAIRS_PER_LABEL * 2:
        raise ValueError('The regional seed list is too small for the requested balanced output.')
    rng = random.Random(13)
    rng.shuffle(seeds)
    jobs = [(seed, 'T') for seed in seeds[:REGIONAL_PAIRS_PER_LABEL]] + [(seed, 'F') for seed in seeds[REGIONAL_PAIRS_PER_LABEL:REGIONAL_PAIRS_PER_LABEL * 2]]
    rng.shuffle(jobs)
    client = OpenAI(api_key=OPENAI_API_KEY)
    rows = []
    for seed, label in tqdm(jobs, desc='Generating regional PT-BR contexts', unit='pair'):
        generated = generate_regional_row(seed, label, client)
        rows.append({
            **generated,
            'target_word': seed['word'],
            'label': label,
            'sentence1_marked': mark_target(generated['sentence1'], seed['word']),
            'sentence2_marked': mark_target(generated['sentence2'], seed['word']),
            'regional_label': seed['regional_label'],
            'regional_level': seed['regional_level'],
            'source_url': seed['source_url'],
            'source_snapshot': seed['source_snapshot'],
            'source_sense_id': seed['sense_id'],
            'generation_method': 'llm_from_kaikki_regional_seed',
            'validation_status': 'needs_linguist_review',
        })
    regional_llm_path = OUTPUT_DIR / 'pipeline2_regional_llm_wic.csv'
    pd.DataFrame(rows).to_csv(regional_llm_path, index=False)
    print(f'Wrote {len(rows)} Pipeline 2 candidates to {regional_llm_path.name}.')
else:
    print('Regional LLM generation is disabled. Set RUN_REGIONAL_LLM_GENERATION = True to run it.')


Regional LLM generation is disabled. Set RUN_REGIONAL_LLM_GENERATION = True to run it.


## Pipeline 3. Indigenous Language--Portuguese Cross-Lingual Candidates

This pipeline collects only public entries from all dictionaries listed on the DicWeb Indigenous platform. Each candidate pairs an authentic example in an Indigenous language with its Portuguese translation. Same-sense pairs come from one lexical sense and different-sense pairs come from distinct senses of the same Indigenous lemma. The output retains dictionary, entry, sense, and source-URL provenance for linguistic and community-based review.


In [6]:
# Pipeline 3: Indigenous language--Portuguese cross-lingual candidates from DicWeb.
# This collection uses DicWeb's public Firestore queries, not HTML scraping.
# Respect the platform and use the output only as validation candidates, with
# linguistic and community review before any release.

import json
import os
import random
import re
import time
import urllib.error
import urllib.parse
import urllib.request
from collections import defaultdict

import pandas as pd
from tqdm.auto import tqdm

DICWEB_API_ROOT = 'https://firestore.googleapis.com/v1/projects/dicwebindigena/databases/(default)/documents:runQuery'
# This is the public browser API key exposed by the DicWeb client application.
DICWEB_FIRESTORE_API_KEY = os.getenv('DICWEB_FIRESTORE_API_KEY', '')
DICWEB_REQUEST_PAUSE_SECONDS = 0.4
RUN_DICWEB_EXTRACTION = True
MAX_PAIRS_PER_LABEL_PER_DICTIONARY = 100

# Both Wari dictionaries are distinct DicWeb projects and remain separate.
DICWEB_DICTIONARIES = {
    'aikana': 'Aikanã',
    'arikapu': 'Arikapú',
    'arua': 'Aruá',
    'cintalarga': 'Cinta Larga',
    'djeoromitxi': 'Djeoromitxi',
    'kanoe': 'Kanoé',
    'karitiana': 'Karitiana',
    'karo-arara': 'Karo-Arara',
    'kawahiwa-amondawa': 'Kawahiwa-Amondawa',
    'kaxarari': 'Kaxarari',
    'makurap': 'Makurap',
    'ofaie': 'Ofaié',
    'oro-win': 'Oro Win',
    'paiter': 'Paiter',
    'tupari': 'Tupari',
    'wajuru': 'Wajuru',
    'wari2': "Wari' (Oro Nao', Oro Eo', Oro At, Oro Yowin)",
    'wari1': "Wari' (Oro Waram, Oro Waram Xijein, Oro Mon, Cao Oro Waje)",
    'zoro': 'Zoró',
}

PORTUGUESE_STOPWORDS = {
    'a', 'as', 'o', 'os', 'de', 'da', 'das', 'do', 'dos', 'e', 'em', 'no', 'na',
    'nos', 'nas', 'um', 'uma', 'para', 'por', 'com', 'que', 'ou', 'se', 'ser',
}

def firestore_value(value):
    if not isinstance(value, dict):
        return value
    scalar_keys = ('stringValue', 'integerValue', 'doubleValue', 'booleanValue', 'timestampValue')
    for key in scalar_keys:
        if key in value:
            result = value[key]
            return int(result) if key == 'integerValue' else result
    if 'nullValue' in value:
        return None
    if 'mapValue' in value:
        return {key: firestore_value(item) for key, item in value['mapValue'].get('fields', {}).items()}
    if 'arrayValue' in value:
        return [firestore_value(item) for item in value['arrayValue'].get('values', [])]
    if 'referenceValue' in value:
        return value['referenceValue']
    return None

def firestore_document(document):
    fields = document.get('fields', {})
    data = {key: firestore_value(value) for key, value in fields.items()}
    data['_document_path'] = document.get('name', '')
    data['_document_id'] = data['_document_path'].rsplit('/', 1)[-1]
    return data

def run_firestore_query(structured_query):
    if not DICWEB_FIRESTORE_API_KEY:
        raise RuntimeError('Set DICWEB_FIRESTORE_API_KEY before running DicWeb extraction.')
    encoded_key = urllib.parse.urlencode({'key': DICWEB_FIRESTORE_API_KEY})
    request = urllib.request.Request(
        f'{DICWEB_API_ROOT}?{encoded_key}',
        data=json.dumps({'structuredQuery': structured_query}).encode('utf-8'),
        headers={'Content-Type': 'application/json', 'User-Agent': 'PORTUWiC-BR research pipeline/0.6'},
        method='POST',
    )
    try:
        with urllib.request.urlopen(request, timeout=90) as response:
            payload = json.load(response)
    except urllib.error.HTTPError as error:
        details = error.read().decode('utf-8', errors='replace')
        raise RuntimeError(f'DicWeb Firestore request failed with HTTP {error.code}: {details[:500]}') from error
    if not isinstance(payload, list):
        raise RuntimeError(f'Unexpected DicWeb Firestore response: {payload}')
    errors = [item['error'] for item in payload if 'error' in item]
    if errors:
        raise RuntimeError(f'DicWeb Firestore query error: {errors[0]}')
    return [firestore_document(item['document']) for item in payload if 'document' in item]

def project_for_slug(slug):
    query = {
        'from': [{'collectionId': 'projetos'}],
        'where': {'fieldFilter': {
            'field': {'fieldPath': 'slug'},
            'op': 'EQUAL',
            'value': {'stringValue': slug},
        }},
    }
    projects = run_firestore_query(query)
    if len(projects) != 1:
        raise RuntimeError(f'Expected one DicWeb project for {slug!r}, found {len(projects)}.')
    return projects[0]

def active_entries_for_project(project_id):
    query = {
        'from': [{'collectionId': 'entradas'}],
        'where': {'compositeFilter': {
            'op': 'AND',
            'filters': [
                {'fieldFilter': {
                    'field': {'fieldPath': 'projetoId'},
                    'op': 'EQUAL',
                    'value': {'stringValue': project_id},
                }},
                {'fieldFilter': {
                    'field': {'fieldPath': 'excluida'},
                    'op': 'EQUAL',
                    'value': {'booleanValue': False},
                }},
            ],
        }},
    }
    return run_firestore_query(query)

def occurrences(sentence, target):
    return len(re.findall(rf'(?<!\w){re.escape(target)}(?!\w)', sentence, flags=re.IGNORECASE))

def marked(sentence, target):
    match = re.search(rf'(?<!\w){re.escape(target)}(?!\w)', sentence, flags=re.IGNORECASE)
    if not match:
        return ''
    return f'{sentence[:match.start()]}<target>{sentence[match.start():match.end()]}</target>{sentence[match.end():]}'

def portuguese_targets(definition, translated_sentence):
    definition = re.sub(r'\s+', ' ', str(definition or '')).strip()
    translated_sentence = str(translated_sentence or '').strip()
    if not definition or not translated_sentence:
        return []
    candidates = []
    for chunk in re.split(r'[;,/()]', definition):
        chunk = chunk.strip(' .:-')
        if chunk and occurrences(translated_sentence, chunk) == 1:
            candidates.append(chunk)
        for token in re.findall(r"[A-Za-zÀ-ÖØ-öø-ÿ][A-Za-zÀ-ÖØ-öø-ÿ'-]*", chunk):
            if token.lower() not in PORTUGUESE_STOPWORDS and occurrences(translated_sentence, token) == 1:
                candidates.append(token)
    return list(dict.fromkeys(candidates))

def attest_senses(slug, language_name, project, entries):
    attested = []
    source_url = f'https://dicweb.app/dicionarios/{slug}/dicionario'
    for entry in entries:
        lemma = str(entry.get('entradaLexical', '')).strip()
        if not lemma:
            continue
        for sense_number, sense in enumerate(entry.get('significados', []) or [], start=1):
            definition = str(sense.get('definicao', '')).strip()
            for example_number, example in enumerate(sense.get('exemplos', []) or [], start=1):
                sentence_indigenous = str(example.get('exemplo', '')).strip()
                sentence_pt = str(example.get('traducao', '')).strip()
                if occurrences(sentence_indigenous, lemma) != 1:
                    continue
                for target_pt in portuguese_targets(definition, sentence_pt):
                    attested.append({
                        'dictionary_slug': slug,
                        'dictionary_name': str(project.get('nome', '')).strip(),
                        'indigenous_language': language_name,
                        'entry_id': entry['_document_id'],
                        'lemma_indigenous': lemma,
                        'sense_number': sense_number,
                        'sense_definition_pt': definition,
                        'example_number': example_number,
                        'target_word_pt': target_pt,
                        'sentence_indigenous': sentence_indigenous,
                        'sentence_pt': sentence_pt,
                        'source_url': source_url,
                    })
    return attested

def candidate_row(left, right, label):
    return {
        'dictionary_slug': left['dictionary_slug'],
        'dictionary_name': left['dictionary_name'],
        'indigenous_language': left['indigenous_language'],
        'label': label,
        'target_word_indigenous': left['lemma_indigenous'],
        'target_word_pt': right['target_word_pt'],
        'sentence_indigenous': left['sentence_indigenous'],
        # This is always the direct, authentic translation of the Indigenous source sentence.
        'sentence_indigenous_translation_pt': left['sentence_pt'],
        'sentence_pt': right['sentence_pt'],
        'sentence_indigenous_marked': marked(left['sentence_indigenous'], left['lemma_indigenous']),
        'sentence_indigenous_translation_pt_marked': marked(left['sentence_pt'], left['target_word_pt']),
        'sentence_pt_marked': marked(right['sentence_pt'], right['target_word_pt']),
        'sense_definition_pt_1': left['sense_definition_pt'],
        'sense_definition_pt_2': right['sense_definition_pt'],
        'entry_id_1': left['entry_id'],
        'entry_id_2': right['entry_id'],
        'sense_number_1': left['sense_number'],
        'sense_number_2': right['sense_number'],
        'example_number_1': left['example_number'],
        'example_number_2': right['example_number'],
        'source_url': left['source_url'],
        'generation_method': 'dicweb_public_bilingual_examples',
        'validation_status': 'needs_linguist_and_community_review',
    }

def build_dicweb_candidates(attested):
    same, different = [], []
    seen_same, seen_different = set(), set()
    by_sense = defaultdict(list)
    for item in attested:
        by_sense[(item['entry_id'], item['sense_number'])].append(item)

    for sense_items in by_sense.values():
        for item in sense_items:
            key = (item['entry_id'], item['sense_number'], item['example_number'], item['target_word_pt'])
            if key not in seen_same:
                same.append(candidate_row(item, item, 'T'))
                seen_same.add(key)

    by_lemma = defaultdict(list)
    for key, sense_items in by_sense.items():
        by_lemma[sense_items[0]['lemma_indigenous'].casefold()].append((key, sense_items))
    for sense_groups in by_lemma.values():
        for left_index, (_, left_items) in enumerate(sense_groups):
            for _, right_items in sense_groups[left_index + 1:]:
                if left_items[0]['sense_definition_pt'].casefold() == right_items[0]['sense_definition_pt'].casefold():
                    continue
                for left in left_items:
                    for right in right_items:
                        key = (left['entry_id'], left['sense_number'], right['sense_number'], left['example_number'], right['example_number'])
                        if key not in seen_different:
                            different.append(candidate_row(left, right, 'F'))
                            seen_different.add(key)
    return same, different

def select_balanced_by_dictionary(candidates, maximum_per_label):
    rng = random.Random(13)
    selected = []
    for slug, group in candidates.groupby('dictionary_slug', sort=True):
        same = group[group['label'] == 'T'].to_dict('records')
        different = group[group['label'] == 'F'].to_dict('records')
        take = min(len(same), len(different), maximum_per_label)
        if take:
            selected.extend(rng.sample(same, take))
            selected.extend(rng.sample(different, take))
    return pd.DataFrame(selected)

if RUN_DICWEB_EXTRACTION:
    all_candidates = []
    coverage_rows = []
    for slug, language_name in tqdm(DICWEB_DICTIONARIES.items(), desc='Collecting DicWeb dictionaries', unit='dictionary'):
        project = project_for_slug(slug)
        time.sleep(DICWEB_REQUEST_PAUSE_SECONDS)
        entries = active_entries_for_project(project['_document_id'])
        time.sleep(DICWEB_REQUEST_PAUSE_SECONDS)
        attested = attest_senses(slug, language_name, project, entries)
        same, different = build_dicweb_candidates(attested)
        all_candidates.extend(same)
        all_candidates.extend(different)
        coverage_rows.append({
            'dictionary_slug': slug,
            'indigenous_language': language_name,
            'active_entries': len(entries),
            'attested_bilingual_examples': len(attested),
            'same_sense_candidates': len(same),
            'different_sense_candidates': len(different),
            'balanced_per_label': min(len(same), len(different), MAX_PAIRS_PER_LABEL_PER_DICTIONARY),
        })

    all_frame = pd.DataFrame(all_candidates)
    all_path = OUTPUT_DIR / 'pipeline3_dicweb_indigenous_pt_wic_all.csv'
    all_frame.to_csv(all_path, index=False)
    balanced_frame = select_balanced_by_dictionary(all_frame, MAX_PAIRS_PER_LABEL_PER_DICTIONARY) if not all_frame.empty else all_frame
    balanced_path = OUTPUT_DIR / 'pipeline3_dicweb_indigenous_pt_wic_balanced.csv'
    balanced_frame.to_csv(balanced_path, index=False)
    coverage_path = OUTPUT_DIR / 'pipeline3_dicweb_indigenous_coverage.csv'
    pd.DataFrame(coverage_rows).to_csv(coverage_path, index=False)
    print(f'Wrote {len(all_frame)} authentic candidates to {all_path.name}.')
    print(f'Wrote {len(balanced_frame)} balanced candidates to {balanced_path.name}.')
    print(f'Wrote per-dictionary coverage to {coverage_path.name}.')
else:
    print('DicWeb extraction is disabled. Set RUN_DICWEB_EXTRACTION = True when ready.')


Wrote 4468 authentic candidates to pipeline3_dicweb_indigenous_pt_wic_all.csv.
Wrote 310 balanced candidates to pipeline3_dicweb_indigenous_pt_wic_balanced.csv.
Wrote per-dictionary coverage to pipeline3_dicweb_indigenous_coverage.csv.


In [7]:
pd.set_option('display.max_columns', None)
pd.read_csv(OUTPUT_DIR / 'pipeline3_dicweb_indigenous_pt_wic_all.csv').indigenous_language.value_counts()


indigenous_language
Wari' (Oro Waram, Oro Waram Xijein, Oro Mon, Cao Oro Waje)    2127
Zoró                                                           667
Paiter                                                         452
Cinta Larga                                                    182
Karitiana                                                      170
Aikanã                                                         167
Karo-Arara                                                     156
Kawahiwa-Amondawa                                              121
Tupari                                                         119
Kaxarari                                                        87
Wari' (Oro Nao', Oro Eo', Oro At, Oro Yowin)                    82
Djeoromitxi                                                     56
Oro Win                                                         47
Kanoé                                                           16
Makurap                                   

In [9]:
pd.read_csv(OUTPUT_DIR / 'pipeline3_dicweb_indigenous_pt_wic_all.csv')


,dictionary_slug,dictionary_name,indigenous_language,label,target_word_indigenous,target_word_pt,sentence_indigenous,sentence_indigenous_translation_pt,sentence_pt,sentence_indigenous_marked,sentence_indigenous_translation_pt_marked,sentence_pt_marked,sense_definition_pt_1,sense_definition_pt_2,entry_id_1,entry_id_2,sense_number_1,sense_number_2,example_number_1,example_number_2,source_url,generation_method,validation_status
0,aikana,Dicionário da Língua Aikanã,Aikanã,T,Haki,Milho,Haki ãupeẽ.,Milho secou.,Milho secou.,<target>Haki</target> ãupeẽ.,<target>Milho</target> secou.,<target>Milho</target> secou.,Milho,Milho,0Sp96OEOhXIoc4FLLAcA,0Sp96OEOhXIoc4FLLAcA,1,1,1,1,https://dicweb.app/dicionarios/aikana/dicionario,dicweb_public_bilingual_examples,needs_linguist_and_community_review
1,aikana,Dicionário da Língua Aikanã,Aikanã,T,Kizu,Bacuri,Derinena kizu kũkɨitakaẽ ama'ete.,Amanhã vou comer bacuri no mato.,Amanhã vou comer bacuri no mato.,Derinena <target>kizu</target> kũkɨitakaẽ ama'...,Amanhã vou comer <target>bacuri</target> no mato.,Amanhã vou comer <target>bacuri</target> no mato.,Bacuri,Bacuri,1P8DoumuBqB0utsXxxcm,1P8DoumuBqB0utsXxxcm,1,1,1,1,https://dicweb.app/dicionarios/aikana/dicionario,dicweb_public_bilingual_examples,needs_linguist_and_community_review
2,aikana,Dicionário da Língua Aikanã,Aikanã,T,Dipa,Lesma,Hisa dipa dapate'ẽ.,Eu vi lesma.,Eu vi lesma.,Hisa <target>dipa</target> dapate'ẽ.,Eu vi <target>lesma</target>.,Eu vi <target>lesma</target>.,Lesma,Lesma,2wVo3P8f9oc9FlVwxo8K,2wVo3P8f9oc9FlVwxo8K,1,1,1,1,https://dicweb.app/dicionarios/aikana/dicionario,dicweb_public_bilingual_examples,needs_linguist_and_community_review
3,aikana,Dicionário da Língua Aikanã,Aikanã,T,Su'a'i,Enxada,Hisa su'a'i zãmẽi amarika kate'ẽ.,Eu comprei enxada nova.,Eu comprei enxada nova.,Hisa <target>su'a'i</target> zãmẽi amarika kat...,Eu comprei <target>enxada</target> nova.,Eu comprei <target>enxada</target> nova.,Enxada,Enxada,32FuH3RO9KZ8ZSEHxCsL,32FuH3RO9KZ8ZSEHxCsL,1,1,1,1,https://dicweb.app/dicionarios/aikana/dicionario,dicweb_public_bilingual_examples,needs_linguist_and_community_review
4,aikana,Dicionário da Língua Aikanã,Aikanã,T,Ewadukɨi,Cacique,Ewadukɨi warerüpeẽ.,Cacique viajou.,Cacique viajou.,<target>Ewadukɨi</target> warerüpeẽ.,<target>Cacique</target> viajou.,<target>Cacique</target> viajou.,Cacique,Cacique,3PzMAgmnNMipYUbUWCMa,3PzMAgmnNMipYUbUWCMa,1,1,1,1,https://dicweb.app/dicionarios/aikana/dicionario,dicweb_public_bilingual_examples,needs_linguist_and_community_review
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4463,zoro,Dicionário da Língua Zoró,Zoró,F,Pawit,Alimentos,Pawit má akunbá.,A comida já acabou.,Vovó mandou o nosso alimentos.,<target>Pawit</target> má akunbá.,A <target>comida</target> já acabou.,Vovó mandou o nosso <target>alimentos</target>.,Comida,Alimentos,1WnhMeqlSn43r3HJtgjn,d72FAmxoDaKNMMu4enrm,1,1,2,2,https://dicweb.app/dicionarios/zoro/dicionario,dicweb_public_bilingual_examples,needs_linguist_and_community_review
4464,zoro,Dicionário da Língua Zoró,Zoró,F,Kujã,Cedro,Kujã mi panga Iwap wetá.,De remo nos guia o barco.,O Cedro é uma árvore linda.,<target>Kujã</target> mi panga Iwap wetá.,De <target>remo</target> nos guia o barco.,O <target>Cedro</target> é uma árvore linda.,Remo,Cedro,PgADUhFZvKcnAwI3spKs,aVJpYypA0XaLMKH7SaqG,1,1,1,1,https://dicweb.app/dicionarios/zoro/dicionario,dicweb_public_bilingual_examples,needs_linguist_and_community_review
4465,zoro,Dicionário da Língua Zoró,Zoró,F,Serewa,Pica pau,Serewa anga mandegaj wá.,Pica-pau come insetos.,Eu vi pica pau numa árvore.,<target>Serewa</target> anga mandegaj wá.,<target>Pica-pau</target> come insetos.,Eu vi <target>pica pau</target> numa árvore.,Pica-pau,Pica pau,ikDNwtabNngMMLU3BNUV,lfGnsFSHTPyRXbU0qRAv,1,1,1,1,https://dicweb.app/dicionarios/zoro/dicionario,dicweb_public_bilingual_examples,needs_linguist_and_community_review
4466,zoro,Dicionário da Língua Zoró,Zoró,F,Xikiripã,Carapanã,Xikirip

### Pipeline 3B. Targeted Portuguese LLM Completion

The selected Indigenous language dictionaries are expanded to 100 same-sense and 100 different-sense candidates each. The Indigenous sentence and its direct DicWeb Portuguese translation are never generated or changed. When an authentic class is scarce, an LLM generates only the second Portuguese context from the registered sense information. Different-sense completions without an attested distinct sense are retained as explicitly unverified proposals rather than gold labels.


In [8]:
# Pipeline 3B: complete selected Indigenous language--Portuguese sets with Portuguese-only LLM contexts.
# Run Pipeline 3 first so that the authentic DicWeb candidate file exists.
# %pip install openai pandas tqdm

import json
import os
import random
import re
import time

import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

try:
    from openai import OpenAI
except ImportError:
    OpenAI = None

SELECTED_INDIGENOUS_LANGUAGES = [
    "Wari' (Oro Waram, Oro Waram Xijein, Oro Mon, Cao Oro Waje)",
    'Zoró',
    'Paiter',
    'Cinta Larga',
    'Karitiana',
    'Aikanã',
    'Karo-Arara',
    'Kawahiwa-Amondawa',
    'Tupari',
]
TARGET_PER_LABEL = 100
DICWEB_COMPLETION_MODEL = 'gpt-4o-mini'
DICWEB_COMPLETION_TEMPERATURE = 0.7
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', '')
# OPENAI_API_KEY = 'paste-your-key-here'
RUN_DICWEB_LLM_COMPLETION = False
LLM_RETRIES = 3

AUTHENTIC_DICWEB_PATH = OUTPUT_DIR / 'pipeline3_dicweb_indigenous_pt_wic_all.csv'
TARGETED_DICWEB_PATH = OUTPUT_DIR / 'pipeline3_dicweb_indigenous_pt_target_100_per_label.csv'
COMPLETION_PLAN_PATH = OUTPUT_DIR / 'pipeline3_dicweb_indigenous_llm_completion_plan.csv'

COMPLETION_SCHEMA = {
    'type': 'object',
    'properties': {
        'sentence_pt': {'type': 'string'},
        'proposed_sense_definition_pt': {'type': 'string'},
    },
    'required': ['sentence_pt', 'proposed_sense_definition_pt'],
    'additionalProperties': False,
}

def exact_occurrences(sentence, target):
    return len(re.findall(rf'(?<!\w){re.escape(target)}(?!\w)', str(sentence), flags=re.IGNORECASE))

def mark_exact_target(sentence, target):
    match = re.search(rf'(?<!\w){re.escape(target)}(?!\w)', str(sentence), flags=re.IGNORECASE)
    if not match:
        return ''
    return f'{sentence[:match.start()]}<target>{sentence[match.start():match.end()]}</target>{sentence[match.end():]}'

def completion_jobs(authentic_frame):
    rng = random.Random(13)
    selected_rows, jobs, plan_rows = [], [], []
    for language in SELECTED_INDIGENOUS_LANGUAGES:
        language_frame = authentic_frame[authentic_frame['indigenous_language'] == language]
        for label in ('T', 'F'):
            candidates = language_frame[language_frame['label'] == label].to_dict('records')
            authentic_count = min(len(candidates), TARGET_PER_LABEL)
            if authentic_count:
                selected_rows.extend(rng.sample(candidates, authentic_count))
            missing = TARGET_PER_LABEL - authentic_count
            fallback_type = 'none'
            if missing:
                if not candidates and label == 'F':
                    # No different-sense evidence exists. Use a same-sense row only as an
                    # Indigenous anchor and flag the proposed Portuguese sense as unattested.
                    candidates = language_frame[language_frame['label'] == 'T'].to_dict('records')
                    fallback_type = 'unattested_different_sense_proposal'
                elif candidates:
                    fallback_type = 'attested_sense_template'
                if not candidates:
                    raise RuntimeError(f'{language} has no authentic examples that can anchor a {label} completion.')
                for _ in range(missing):
                    jobs.append({
                        'template': rng.choice(candidates),
                        'label': label,
                        'fallback_type': fallback_type,
                    })
            plan_rows.append({
                'indigenous_language': language,
                'label': label,
                'authentic_available': len(language_frame[language_frame['label'] == label]),
                'authentic_selected': authentic_count,
                'llm_to_generate': missing,
                'completion_evidence': fallback_type,
            })
    return selected_rows, jobs, pd.DataFrame(plan_rows)

def generate_portuguese_completion(job, client):
    template = job['template']
    label = job['label']
    is_unattested_f = job['fallback_type'] == 'unattested_different_sense_proposal'
    if label == 'T':
        instruction = (
            'Write a new natural Brazilian Portuguese sentence that uses the target in the same sense '
            'as the registered sense. Do not reproduce the reference translation.'
        )
        second_definition = template['sense_definition_pt_1']
    elif is_unattested_f:
        instruction = (
            'Write a new natural Brazilian Portuguese sentence in a clearly different common sense of the '
            'Portuguese target. This is only a candidate hypothesis. Do not claim that the Indigenous lemma '
            'has this second sense and do not reproduce the reference translation.'
        )
        second_definition = ''
    else:
        instruction = (
            'Write a new natural Brazilian Portuguese sentence using the target in registered sense 2, which '
            'is distinct from sense 1. Do not reproduce either reference translation.'
        )
        second_definition = template['sense_definition_pt_2']
    prompt = f"""
Create one Portuguese context for a cross-lingual WiC validation candidate.

Indigenous language: {template['indigenous_language']}
Indigenous lemma: {template['target_word_indigenous']}
Authentic Indigenous sentence: {template['sentence_indigenous']}
Direct Portuguese translation of that Indigenous sentence: {template['sentence_indigenous_translation_pt']}
Portuguese target: {template['target_word_pt']}
Registered sense 1: {template['sense_definition_pt_1']}
Registered sense 2: {second_definition or '[no registered second sense]'}

{instruction}
Use the exact Portuguese target exactly once. Return no Indigenous-language text, no explanations, and no target tags.
""".strip()
    last_error = None
    for attempt in range(LLM_RETRIES):
        try:
            response = client.responses.create(
                model=DICWEB_COMPLETION_MODEL,
                temperature=DICWEB_COMPLETION_TEMPERATURE,
                input=[
                    {'role': 'developer', 'content': 'Return only valid JSON. Do not invent or alter Indigenous-language content.'},
                    {'role': 'user', 'content': prompt},
                ],
                text={'format': {'type': 'json_schema', 'name': 'dicweb_pt_completion', 'strict': True, 'schema': COMPLETION_SCHEMA}},
            )
            generated = json.loads(response.output_text)
            sentence = generated['sentence_pt'].strip()
            if exact_occurrences(sentence, template['target_word_pt']) != 1:
                raise ValueError('The generated Portuguese target must occur exactly once.')
            return generated
        except Exception as error:
            last_error = error
            time.sleep(1 + attempt)
    raise RuntimeError(f'LLM completion failed after {LLM_RETRIES} attempts.') from last_error

if not AUTHENTIC_DICWEB_PATH.exists():
    raise FileNotFoundError('Run Pipeline 3 before Pipeline 3B.')

authentic_frame = pd.read_csv(AUTHENTIC_DICWEB_PATH).fillna('')
selected_rows, jobs, completion_plan = completion_jobs(authentic_frame)
completion_plan.to_csv(COMPLETION_PLAN_PATH, index=False)
print('LLM completion plan')
display(completion_plan)
print(f'Planned Portuguese-only LLM generations: {len(jobs)}.')

if RUN_DICWEB_LLM_COMPLETION:
    if OpenAI is None or not OPENAI_API_KEY:
        raise RuntimeError('Set OPENAI_API_KEY and install the OpenAI SDK before running Pipeline 3B.')
    client = OpenAI(api_key=OPENAI_API_KEY)
    completed_rows = []
    for job in tqdm(jobs, desc='Generating Portuguese-only DicWeb completions', unit='candidate'):
        generated = generate_portuguese_completion(job, client)
        row = dict(job['template'])
        row['label'] = job['label']
        row['sentence_pt'] = generated['sentence_pt'].strip()
        row['sentence_pt_marked'] = mark_exact_target(row['sentence_pt'], row['target_word_pt'])
        if job['label'] == 'T':
            row['sense_definition_pt_2'] = row['sense_definition_pt_1']
        elif job['fallback_type'] == 'unattested_different_sense_proposal':
            row['sense_definition_pt_2'] = generated['proposed_sense_definition_pt'].strip()
            row['entry_id_2'] = ''
            row['sense_number_2'] = ''
            row['example_number_2'] = ''
        row['sentence_pt_origin'] = 'llm_generated'
        row['completion_evidence'] = job['fallback_type']
        row['llm_model'] = DICWEB_COMPLETION_MODEL
        row['llm_temperature'] = DICWEB_COMPLETION_TEMPERATURE
        row['generation_method'] = 'dicweb_authentic_indigenous_plus_llm_portuguese_context'
        row['validation_status'] = 'needs_linguist_and_community_review'
        completed_rows.append(row)

    authentic_selected = pd.DataFrame(selected_rows)
    authentic_selected['sentence_pt_origin'] = 'dicweb_authentic_translation_or_attested_context'
    authentic_selected['completion_evidence'] = 'fully_attested'
    authentic_selected['llm_model'] = ''
    authentic_selected['llm_temperature'] = ''
    targeted_frame = pd.concat([authentic_selected, pd.DataFrame(completed_rows)], ignore_index=True)
    targeted_frame = targeted_frame.sample(frac=1, random_state=13).reset_index(drop=True)
    expected_size = len(SELECTED_INDIGENOUS_LANGUAGES) * 2 * TARGET_PER_LABEL
    if len(targeted_frame) != expected_size:
        raise RuntimeError(f'Expected {expected_size} rows, wrote {len(targeted_frame)}.')
    targeted_frame.to_csv(TARGETED_DICWEB_PATH, index=False)
    print(f'Wrote {len(targeted_frame)} targeted candidates to {TARGETED_DICWEB_PATH.name}.')
else:
    print('LLM completion is disabled. Review the plan, then set RUN_DICWEB_LLM_COMPLETION = True to create the 1,800-row targeted file.')


LLM completion plan


,indigenous_language,label,authentic_available,authentic_selected,llm_to_generate,completion_evidence
0,"Wari' (Oro Waram, Oro Waram Xijein, Oro Mon, C...",T,1981,100,0,none
1,"Wari' (Oro Waram, Oro Waram Xijein, Oro Mon, C...",F,146,100,0,none
2,Zoró,T,661,100,0,none
3,Zoró,F,6,6,94,attested_sense_template
4,Paiter,T,443,100,0,none
5,Paiter,F,9,9,91,attested_sense_template
6,Cinta Larga,T,182,100,0,none
7,Cinta Larga,F,0,0,100,unattested_different_sense_proposal
8,Karitiana,T,167,100,0,none
9,Karitiana,F,3,3,97,attested_sense_template


Planned Portuguese-only LLM generations: 748.
LLM completion is disabled. Review the plan, then set RUN_DICWEB_LLM_COMPLETION = True to create the 1,800-row targeted file.


## Final Consolidation

The final stage combines the two monolingual pipelines and publishes the balanced Indigenous language--Portuguese resource separately. The complete DicWeb candidate file is retained for audit and subsequent expert review.


In [ ]:
# Final consolidation keeps monolingual and cross-lingual data in separate files.

import random

import pandas as pd

normal_path = OUTPUT_DIR / 'pipeline1_normal_ptbr_wic.csv'
regional_path = OUTPUT_DIR / 'pipeline2_regional_llm_wic.csv'
cross_path = OUTPUT_DIR / 'pipeline3_dicweb_indigenous_pt_target_100_per_label.csv'

monolingual_frames = []
for path, method in [(normal_path, 'pipeline1_normal_ptbr'), (regional_path, 'pipeline2_regional_llm')]:
    if not path.exists():
        print(f'Skipping missing output: {path.name}')
        continue
    frame = pd.read_csv(path)
    frame['pipeline'] = method
    monolingual_frames.append(frame)

if monolingual_frames:
    final_mono = pd.concat(monolingual_frames, ignore_index=True)
    final_mono = final_mono.drop_duplicates(subset=['target_word', 'label', 'sentence1', 'sentence2'])
    final_mono['validation_status'] = final_mono.get('validation_status', 'needs_linguist_review')
    final_mono_path = OUTPUT_DIR / 'portuwic_br_monolingual_candidates.csv'
    final_mono.to_csv(final_mono_path, index=False)
    split_rows = []
    rng = random.Random(13)
    for label, group in final_mono.groupby('label'):
        indices = list(group.index)
        rng.shuffle(indices)
        train_end = int(len(indices) * 0.8)
        dev_end = train_end + int(len(indices) * 0.1)
        split_rows.extend((index, 'train') for index in indices[:train_end])
        split_rows.extend((index, 'dev') for index in indices[train_end:dev_end])
        split_rows.extend((index, 'test') for index in indices[dev_end:])
    final_split = final_mono.copy()
    final_split['split'] = ''
    for index, split in split_rows:
        final_split.loc[index, 'split'] = split
    final_split_path = OUTPUT_DIR / 'portuwic_br_monolingual_splits.csv'
    final_split.to_csv(final_split_path, index=False)
    print(f'Wrote {len(final_mono)} consolidated candidates and splits to {final_split_path.name}.')
else:
    print('No monolingual pipeline outputs are available yet.')

if cross_path.exists():
    final_cross = pd.read_csv(cross_path)
    final_cross['pipeline'] = 'pipeline3_dicweb_indigenous_pt'
    final_cross_path = OUTPUT_DIR / 'portuwic_br_crosslingual_indigenous_pt_candidates.csv'
    final_cross.to_csv(final_cross_path, index=False)
    print(f'Wrote {len(final_cross)} consolidated cross-lingual candidates to {final_cross_path.name}.')
else:
    print('No cross-lingual pipeline output is available yet.')
